# Compare the 75,000-row trial

Verifies the four common-test run packages, computes paired uncertainty, creates comparison figures, and writes a review bundle to the separate trial folder.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
SOURCE_EXPERIMENT_NAME='lending_club_900k'
TRIAL_EXPERIMENT_NAME='lending_club_75k_trial'
RUN_TESTS=True
BOOTSTRAP_RESAMPLES=2000
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
from tfmbench.trial import TrialConfig, verify_trial_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
config=TrialConfig(DRIVE_OUTPUT_DIR,TRIAL_EXPERIMENT_NAME,SOURCE_EXPERIMENT_NAME)
manifest=verify_trial_manifest(config); print('Verified trial manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.trial import build_trial_performance_review
review=build_trial_performance_review(DRIVE_OUTPUT_DIR,SOURCE_EXPERIMENT_NAME,TRIAL_EXPERIMENT_NAME,bootstrap_resamples=BOOTSTRAP_RESAMPLES)


In [ ]:
import pandas as pd
print(review.comparison_path.read_text())
display(pd.read_csv(review.review_dir/'all_metrics.csv'))
display(review.inventory)
print('Review bundle:',review.review_dir/'review_bundle.zip')
